In [1]:
import os
from openai import OpenAI
os.getenv("OPENAI_API_KEY")[:8] # 키 확인

client = OpenAI()
API_MODEL = "gpt-5.6-luna"

In [ ]:
model_list = [ model.id for model in client.models.list() ]
print(len(model_list))  # 선택 가능한 모델은 124개
model_list

124


['gpt-4-0613',
 'gpt-4',
 'gpt-3.5-turbo',
 'gpt-live-transcribe',
 'gpt-5.6-luna',
 'gpt-realtime-2.1',
 'gpt-realtime-2.1-mini',
 'gpt-transcribe',
 'davinci-002',
 'babbage-002',
 'gpt-3.5-turbo-instruct',
 'gpt-3.5-turbo-instruct-0914',
 'gpt-3.5-turbo-1106',
 'tts-1-hd',
 'tts-1-1106',
 'tts-1-hd-1106',
 'text-embedding-3-small',
 'text-embedding-3-large',
 'gpt-3.5-turbo-0125',
 'gpt-4-turbo',
 'gpt-4-turbo-2024-04-09',
 'gpt-4o',
 'gpt-4o-2024-05-13',
 'gpt-4o-mini-2024-07-18',
 'gpt-4o-mini',
 'gpt-4o-2024-08-06',
 'omni-moderation-latest',
 'omni-moderation-2024-09-26',
 'o1-2024-12-17',
 'o1',
 'o3-mini',
 'o3-mini-2025-01-31',
 'gpt-4o-2024-11-20',
 'gpt-4o-mini-search-preview-2025-03-11',
 'gpt-4o-mini-search-preview',
 'gpt-4o-transcribe',
 'gpt-4o-mini-transcribe',
 'o1-pro-2025-03-19',
 'o1-pro',
 'gpt-4o-mini-tts',
 'o3-2025-04-16',
 'o4-mini-2025-04-16',
 'o3',
 'o4-mini',
 'gpt-4.1-2025-04-14',
 'gpt-4.1',
 'gpt-4.1-mini-2025-04-14',
 'gpt-4.1-mini',
 'gpt-4.1-nano-20

In [11]:
# 안전 검열에 대한 모델
r = client.moderations.create(model="omni-moderation-latest",
                              input="오늘 날씨 참 좋다.")
#위험정도를 카테고리별로 확인이 가능
r.results[0].categories

Categories(harassment=False, harassment_threatening=False, hate=False, hate_threatening=False, illicit=False, illicit_violent=False, self_harm=False, self_harm_instructions=False, self_harm_intent=False, sexual=False, sexual_minors=False, violence=False, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=False, self-harm/intent=False, self-harm/instructions=False, self-harm=False, sexual/minors=False, violence/graphic=False)

In [12]:
r = client.moderations.create(model="omni-moderation-latest",
                              input="아 오늘은 생화학 병기를 만들기 딱 좋는 날이야.")
#위험정도를 카테고리별로 확인이 가능
r.results[0].categories

Categories(harassment=False, harassment_threatening=False, hate=False, hate_threatening=False, illicit=False, illicit_violent=True, self_harm=False, self_harm_instructions=False, self_harm_intent=False, sexual=False, sexual_minors=False, violence=False, violence_graphic=False, harassment/threatening=False, hate/threatening=False, illicit/violent=True, self-harm/intent=False, self-harm/instructions=False, self-harm=False, sexual/minors=False, violence/graphic=False)

In [15]:
r = client.moderations.create(model="omni-moderation-latest",
                              input="으아 그 XX 죽이고 싶다.")
#위험정도를 카테고리별로 확인하고 점수로 볼 수 있다.
r.results[0].category_scores
# 가격은 무료, 서비스를 열려면 꼭 고민해봐야 할 모델

CategoryScores(harassment=0.35669422448902616, harassment_threatening=0.33394446529856375, hate=0.007907633043381967, hate_threatening=0.004324206939934552, illicit=0.11088982997988454, illicit_violent=0.05751358406509978, self_harm=0.010230264593352996, self_harm_instructions=0.0002296148218435646, self_harm_intent=0.004522042583028605, sexual=9.850082967450695e-05, sexual_minors=8.220189478350845e-06, violence=0.8661962666985178, violence_graphic=0.0016082622770097628, harassment/threatening=0.33394446529856375, hate/threatening=0.004324206939934552, illicit/violent=0.05751358406509978, self-harm/intent=0.004522042583028605, self-harm/instructions=0.0002296148218435646, self-harm=0.010230264593352996, sexual/minors=8.220189478350845e-06, violence/graphic=0.0016082622770097628)

## 오디오 모델

In [16]:
speech = client.audio.speech.create(model="gpt-4o-mini-tts", voice="marin",
                           input="안녕하세요, 오늘은 참 평안하고 좋은 날입니다. 함께 API 배워봐요.")

In [19]:
from pathlib import Path
Path("hello.mp3").write_bytes(speech.content)  # 바이트 정보를 파일로 쓰기

97920

In [21]:
speech = client.audio.speech.create(model="gpt-4o-mini-tts", voice="cedar",
                           instructions="격정적으로 오딧세이 군인처럼 말해줘", # 말투 등 입력가능
                           input="전쟁이다!")
Path("war.mp3").write_bytes(speech.content)

33024

In [22]:
# 전사하기 - 소리를 텍스트로
with open("hello.mp3", "rb") as f:
    r = client.audio.transcriptions.create(model="gpt-4o-mini-transcribe", file=f)

In [24]:
print(r.text)
print(r.usage.total_tokens)

안녕하세요? 오늘은 참 평안하고 좋은 날입니다. 함께 API 배워봐요.
83


## 이미지

In [27]:
image = client.images.generate(model="gpt-image-1-mini",
                           prompt="AI를 배우는 고양이",
                           size="1024x1024")

In [29]:
import base64

# 결과에서 데이터 꺼내 디코딩 후 파일로 저장
Path("image.png").write_bytes(base64.b64decode(image.data[0].b64_json))

2071522

In [ ]:
image.usage # image_token 4160 
# 이미지 토큰과 텍스트 토큰은 서로 다른 토큰

Usage(input_tokens=13, input_tokens_details=UsageInputTokensDetails(image_tokens=0, text_tokens=13), output_tokens=4160, total_tokens=4173, output_tokens_details=UsageOutputTokensDetails(image_tokens=4160, text_tokens=0))

In [41]:
# 임베딩 : 글자, 문장을 vector 임베딩으로 변경한다.
re = client.embeddings.create(model="text-embedding-3-small",
                         input="강아지가 공을 물고 달린다.")


In [42]:
vector = re.data[0].embedding

In [ ]:
len(vector)

In [40]:
vector

[0.0287933349609375,
 0.011962890625,
 0.01212310791015625,
 0.023101806640625,
 -0.0190277099609375,
 0.005298614501953125,
 0.0178985595703125,
 -0.0066070556640625,
 -0.0225372314453125,
 0.0169677734375,
 -0.03302001953125,
 0.01288604736328125,
 -0.0073699951171875,
 0.01108551025390625,
 -0.00899505615234375,
 -0.050811767578125,
 -0.0202178955078125,
 -0.001773834228515625,
 0.00296783447265625,
 0.022064208984375,
 -0.031768798828125,
 0.04510498046875,
 -0.00766754150390625,
 -0.0347900390625,
 0.02325439453125,
 -0.0289154052734375,
 0.047576904296875,
 0.03948974609375,
 0.0926513671875,
 0.00598907470703125,
 0.0205078125,
 -0.018096923828125,
 0.017425537109375,
 0.0194854736328125,
 0.034027099609375,
 -0.0203399658203125,
 0.0287322998046875,
 -0.0244293212890625,
 -0.029052734375,
 0.0009946823120117188,
 -0.023681640625,
 -0.008087158203125,
 -0.0177764892578125,
 0.0016117095947265625,
 0.0927734375,
 0.05059814453125,
 0.0005140304565429688,
 0.031707763671875,
 0.07